# Assignment 2: Word Counting and Analysis of text


## Install and initialize

In [ ]:
!pip install pyspark
from pyspark.sql import SparkSession

# Create a Spark Context
sc = SparkSession.builder.master("local[*]").appName("Test").getOrCreate().sparkContext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.2-py2.py3-none-any.whl size=317812365 sha256=5b8bd2bc1c872ba0914bd0819adc338d86d514a50b1579109672d675cc292616
  Stored in directory: /root/.cache/pip/wheels/34/34/bd/03944534c44b677cd5859f248090daa9fb27b3c8f8e5f49574
Successfully built pyspark


## Word Counting Task
### Finish the following tasks
1. Start from the provided text, create an RDD named `words` that contains all words in lower case for future use
2. Use `words` RDD to generate a word counts list in the form of pairs like ``[('apple', 32), ('banana', 21), ... ]`` and store it in an RDD named `wordCounts`
   - Refer to the example notebook in this module for demonstration
   - You will be using `map`, `flatMap`, `reduceByKey` methods to achieve the goal

**Hint**: Use the `take` method to show the first 10 records in both `words` and `wordCounts` RDD at the end of each task to show what you get

In [ ]:
text = '''word count from Wikipedia the free encyclopedia
the word count is the number of words in a document or passage of text Word counting may be needed when a text
is required to stay within certain numbers of words This may particularly be the case in academia legal
proceedings journalism and advertising Word count is commonly used by translators to determine the price for
the translation job Word counts may also be used to calculate measures of readability and to measure typing
and reading speeds usually in words per minute When converting character counts to words a measure of five or
six characters to a word is generally used Contents Details and variations of definition Software In fiction
In non fiction See also References Sources External links Details and variations of definition
This section does not cite any references or sources Please help improve this section by adding citations to
reliable sources Unsourced material may be challenged and removed
Variations in the operational definitions of how to count the words can occur namely what counts as a word and
which words don't count toward the total However especially since the advent of widespread word processing there
is a broad consensus on these operational definitions and hence the bottom line integer result
The consensus is to accept the text segmentation rules generally found in most word processing software including how
word boundaries are determined which depends on how word dividers are defined The first trait of that definition is that a space any of various whitespace
characters such as a regular word space an em space or a tab character is a word divider Usually a hyphen or a slash is too
Different word counting programs may give varying results depending on the text segmentation rule
details and on whether words outside the main text such as footnotes endnotes or hidden text) are counted But the behavior
of most major word processing applications is broadly similar However during the era when school assignments were done in
handwriting or with typewriters the rules for these definitions often differed from todays consensus
Most importantly many students were drilled on the rule that certain words don't count usually articles namely a an the but
sometimes also others such as conjunctions for example and or but and some prepositions usually to of Hyphenated permanent
compounds such as follow up noun or long term adjective were counted as one word To save the time and effort of counting
word by word often a rule of thumb for the average number of words per line was used such as 10 words per line These rules
have fallen by the wayside in the word processing era the word count feature of such software which follows the text
segmentation rules mentioned earlier is now the standard arbiter because it is largely consistent across documents and
applications and because it is fast effortless and costless already included with the application As for which sections of
a document count toward the total such as footnotes endnotes abstracts reference lists and bibliographies tables figure
captions hidden text the person in charge teacher client can define their choice and users students workers can simply
select or exclude the elements accordingly and watch the word count automatically update Software Modern web browsers
support word counting via extensions via a JavaScript bookmarklet or a script that is hosted in a website Most word
processors can also count words Unix like systems include a program wc specifically for word counting
As explained earlier different word counting programs may give varying results depending on the text segmentation rule
details The exact number of words often is not a strict requirement thus the variation is acceptable
In fiction Novelist Jane Smiley suggests that length is an important quality of the novel However novels can vary
tremendously in length Smiley lists novels as typically being between and words while National Novel Writing Month
requires its novels to be at least words There are no firm rules for example the boundary between a novella and a novel
is arbitrary and a literary work may be difficult to categorise But while the length of a novel is to a large extent up
to its writer lengths may also vary by subgenre many chapter books for children start at a length of about words and a
typical mystery novel might be in the to word range while a thriller could be over words
The Science Fiction and Fantasy Writers of America specifies word lengths for each category of its Nebula award categories
Classification	Word count Novel over words Novella to words Novelette to words Short story under words
In non fiction The acceptable length of an academic dissertation varies greatly dependent predominantly on the subject
Numerous American universities limit Ph.D. dissertations to at most words barring special permission for exceeding this limit
'''

In [ ]:
# Write your code here for task 2
# Example output: [('count', 11), ('wikipedia', 1), ('free', 1), ('is', 19), ('of', 25), ('in', 15), ('counting', 6), ('may', 8), ('needed', 1), ('when', 3)]

# Split by new line character and parallelize
lines = sc.parallelize(text.split("\n"))

# Collection of lines first 10
print(lines.take(10))


# Use flatMap to create an RDD of individual words
words = lines.flatMap(lambda line: line.split()) # Changed to flatMap to flatten the lists of words

print('\ncounts of words first 10: ')
# Convert the defaultdict to a list of (key, value) pairs
counts = words.countByValue().items()
# Parallelize the counts to create an RDD
wordCounts = sc.parallelize(counts)
# Take the first 10 elements from the RDD
print(wordCounts.take(10))






['word count from Wikipedia the free encyclopedia', 'the word count is the number of words in a document or passage of text Word counting may be needed when a text', 'is required to stay within certain numbers of words This may particularly be the case in academia legal', 'proceedings journalism and advertising Word count is commonly used by translators to determine the price for', 'the translation job Word counts may also be used to calculate measures of readability and to measure typing', 'and reading speeds usually in words per minute When converting character counts to words a measure of five or', 'six characters to a word is generally used Contents Details and variations of definition Software In fiction', 'In non fiction See also References Sources External links Details and variations of definition', 'This section does not cite any references or sources Please help improve this section by adding citations to', 'reliable sources Unsourced material may be challenged and removed']


3. Use the `words` and `wordCounts` RDD

   A. Calculate the average length of words in the `words` RDD

   B. Find the top 5 longest words
   
   C. Find the top 5 words with the highest counts

Sample run:
```
Avg length: 5.076543209876541
Top longest words: ['bibliographies', 'classification', 'automatically', 'predominantly', 'dissertations']
Top frequent words: [('the', 43), ('word', 28), ('a', 28), ('of', 25), ('and', 23)]
```

In [ ]:
# Write your code here for task 3

# A. Calculate the average length of words in the words RDD
wordlengths = words.map(lambda word: len(word))
averagewordlength = wordlengths.mean()
print('Average word length: ', averagewordlength)

# B. Find the top 5 longest words
top5longestwords = words.map(lambda word: (word, len(word))).top(5, key=lambda x: x[1])
print('Top 5 longest words: ', top5longestwords)

# C. Find the top 5 words with the highest counts
top5wordswithhighestcounts = wordCounts.top(5, key=lambda x: x[1])
print('Top 5 words with highest counts: ', top5wordswithhighestcounts)

Average word length:  5.076543209876541
Top 5 longest words:  [('bibliographies', 14), ('Classification', 14), ('automatically', 13), ('predominantly', 13), ('dissertations', 13)]
Top 5 words with highest counts:  [('the', 38), ('a', 28), ('of', 25), ('word', 24), ('and', 23)]


## Submission

1. After you have finished you code, run and get the correct output.
2. Print the page as PDF and upload the PDF file to Canvas